# Caso MPC Cart-Pole — Análisis del Dataset y Estabilización

Este notebook corresponde a la **primera etapa del caso MPC**:

1. Cargar `X.csv`, `U.csv` y `X0.csv` sin cargar el dataset completo en RAM.
2. Verificar la estructura de las trayectorias.
3. Reconstruir:
   - `X`: 26 estados de 4 variables por trayectoria.
   - `U`: 25 acciones de control por trayectoria.
   - `X0`: condición inicial de 4 estados.
4. Graficar la evolución temporal de los cuatro estados y la acción de control.
5. Calcular métricas básicas de estabilización.
6. Dejar preparado el flujo para comparar posteriormente:
   - MPC experto
   - NN Float32
   - PTQ INT8
   - QAT INT8
   - INT16x8 / Mixed Precision

**Estructura identificada en las muestras:**
- `X0`: 4 valores por trayectoria.
- `X`: 104 valores = 26 × 4 estados.
- `U`: 25 valores.
- Horizonte MPC: `N = 25`.

In [ ]:
from pathlib import Path
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.max_columns", 20)

print("Entorno cargado correctamente.")

## 1. Configuración de rutas

In [ ]:
# Cambia esta ruta por la carpeta donde descomprimiste FinalAMPCPendulumDataset.zip
DATASET_DIR = Path(r"C:\RUTA\AL\FinalAMPCPendulumDataset")

X_PATH  = DATASET_DIR / "X.csv"
U_PATH  = DATASET_DIR / "U.csv"
X0_PATH = DATASET_DIR / "X0.csv"

# Prueba opcional con los mini archivos entregados.
USE_MINI_FILES = False
MINI_DIR = Path("/mnt/data")

if USE_MINI_FILES:
    X_PATH  = MINI_DIR / "X.txt"
    U_PATH  = MINI_DIR / "U.txt"
    X0_PATH = MINI_DIR / "x0.txt"

print("X :", X_PATH)
print("U :", U_PATH)
print("X0:", X0_PATH)

## 2. Carga eficiente de trayectorias

Los CSV originales ocupan varios GB. Por ello, el notebook **no carga el dataset completo en RAM**.
La función siguiente recorre el archivo secuencialmente y conserva únicamente las filas solicitadas.

In [ ]:
def cargar_filas_csv(path, indices, dtype=np.float64):
    path = Path(path)
    indices = list(map(int, indices))
    wanted = set(indices)
    found = {}

    with path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            if i in wanted:
                found[i] = np.asarray(row, dtype=dtype)
                if len(found) == len(wanted):
                    break

    faltantes = [i for i in indices if i not in found]
    if faltantes:
        raise IndexError(
            f"No se encontraron las filas {faltantes} en {path.name}."
        )

    return np.vstack([found[i] for i in indices])


def cargar_trayectorias(indices):
    X_flat = cargar_filas_csv(X_PATH, indices)
    U = cargar_filas_csv(U_PATH, indices)
    X0 = cargar_filas_csv(X0_PATH, indices)

    if X_flat.shape[1] % 4 != 0:
        raise ValueError(
            f"X tiene {X_flat.shape[1]} columnas; no es divisible por 4."
        )

    n_estados_temporales = X_flat.shape[1] // 4
    X = X_flat.reshape(len(indices), n_estados_temporales, 4)

    return X, U, X0

## 3. Cargar algunas trayectorias

In [ ]:
INDICES = [0, 1, 2, 3, 4]

X, U, X0 = cargar_trayectorias(INDICES)

print("Forma X  :", X.shape)
print("Forma U  :", U.shape)
print("Forma X0 :", X0.shape)

print("\nInterpretación:")
print(f"- Trayectorias cargadas   : {X.shape[0]}")
print(f"- Estados por trayectoria : {X.shape[1]}")
print(f"- Variables de estado     : {X.shape[2]}")
print(f"- Controles por trayectoria: {U.shape[1]}")
print(f"- Horizonte inferido N    : {U.shape[1]}")

## 4. Verificación estructural del dataset

In [ ]:
errores_x0 = np.linalg.norm(X[:, 0, :] - X0, axis=1)

df_check = pd.DataFrame({
    "trayectoria": INDICES,
    "||X[0]-X0||": errores_x0
})

display(df_check)

print("\nMáximo error entre X[:,0,:] y X0:")
print(np.max(errores_x0))

assert X.shape[2] == 4, "Se esperaban 4 variables de estado."
assert X.shape[1] == U.shape[1] + 1, (
    "Para MPC debería cumplirse número_estados = número_controles + 1."
)

print("\n✓ Estructura consistente con un horizonte MPC.")

## 5. Convención de variables de estado

Usaremos inicialmente:

\[
x_k =
\begin{bmatrix}
x \\
\dot{x} \\
\theta \\
\dot{\theta}
\end{bmatrix}
\]

donde:
- \(x\): posición del carro.
- \(\dot{x}\): velocidad del carro.
- \(\theta\): ángulo del péndulo.
- \(\dot{\theta}\): velocidad angular.

> Si el repositorio usa otro orden exacto, se debe ajustar aquí antes de interpretar físicamente los gráficos.

In [ ]:
NOMBRES_ESTADO = [
    "Posición x",
    "Velocidad ẋ",
    "Ángulo θ",
    "Velocidad angular θ̇",
]

## 6. Graficar una trayectoria MPC

In [ ]:
def graficar_trayectoria(X, U, pos=0, dt=1.0):
    trayectoria = X[pos]
    control = U[pos]

    t_x = np.arange(trayectoria.shape[0]) * dt
    t_u = np.arange(control.shape[0]) * dt

    for j in range(4):
        plt.figure(figsize=(9, 4))
        plt.plot(t_x, trayectoria[:, j], marker="o", markersize=3)
        plt.axhline(0, linewidth=1)
        plt.xlabel("Tiempo" if dt != 1.0 else "Paso k")
        plt.ylabel(NOMBRES_ESTADO[j])
        plt.title(f"Trayectoria global {INDICES[pos]} — {NOMBRES_ESTADO[j]}")
        plt.grid(True, alpha=0.3)
        plt.show()

    plt.figure(figsize=(9, 4))
    plt.step(t_u, control, where="post")
    plt.axhline(0, linewidth=1)
    plt.xlabel("Tiempo" if dt != 1.0 else "Paso k")
    plt.ylabel("u")
    plt.title(f"Trayectoria global {INDICES[pos]} — Acción de control MPC")
    plt.grid(True, alpha=0.3)
    plt.show()


graficar_trayectoria(X, U, pos=0, dt=1.0)

## 7. Comparar varias trayectorias

In [ ]:
def comparar_trayectorias(X, U, indices_globales, dt=1.0):
    t_x = np.arange(X.shape[1]) * dt
    t_u = np.arange(U.shape[1]) * dt

    for j in range(4):
        plt.figure(figsize=(10, 4))
        for i, idx_global in enumerate(indices_globales):
            plt.plot(t_x, X[i, :, j], label=f"Trayectoria {idx_global}")
        plt.axhline(0, linewidth=1)
        plt.xlabel("Tiempo" if dt != 1.0 else "Paso k")
        plt.ylabel(NOMBRES_ESTADO[j])
        plt.title(f"Comparación — {NOMBRES_ESTADO[j]}")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()

    plt.figure(figsize=(10, 4))
    for i, idx_global in enumerate(indices_globales):
        plt.step(t_u, U[i], where="post", label=f"Trayectoria {idx_global}")
    plt.axhline(0, linewidth=1)
    plt.xlabel("Tiempo" if dt != 1.0 else "Paso k")
    plt.ylabel("u")
    plt.title("Comparación de acciones de control MPC")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


comparar_trayectorias(X, U, INDICES)

## 8. Norma del estado y convergencia al equilibrio

In [ ]:
def graficar_norma_estado(X, indices_globales, dt=1.0):
    t = np.arange(X.shape[1]) * dt

    plt.figure(figsize=(10, 4))
    for i, idx_global in enumerate(indices_globales):
        norma = np.linalg.norm(X[i], axis=1)
        plt.plot(t, norma, label=f"Trayectoria {idx_global}")

    plt.xlabel("Tiempo" if dt != 1.0 else "Paso k")
    plt.ylabel(r"$||x_k||_2$")
    plt.title("Convergencia del estado hacia el equilibrio")
    plt.yscale("log")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


graficar_norma_estado(X, INDICES)

## 9. Métricas básicas de estabilización

Se calculan:
- norma inicial del estado,
- norma final,
- reducción relativa,
- máximo ángulo absoluto,
- máximo desplazamiento absoluto,
- esfuerzo de control \(\sum u_k^2\),
- variación total del control \(\sum |u_k-u_{k-1}|\).

La variación total será especialmente útil después para detectar oscilaciones o *chattering* inducidos por cuantización.

In [ ]:
def metricas_trayectoria(x, u):
    norma = np.linalg.norm(x, axis=1)
    norma_inicial = norma[0]
    norma_final = norma[-1]

    reduccion = np.nan
    if norma_inicial > 0:
        reduccion = 100.0 * (1.0 - norma_final / norma_inicial)

    return {
        "norma_inicial": norma_inicial,
        "norma_final": norma_final,
        "reduccion_%": reduccion,
        "max_abs_x": np.max(np.abs(x[:, 0])),
        "max_abs_theta": np.max(np.abs(x[:, 2])),
        "esfuerzo_control_sum_u2": np.sum(u**2),
        "variacion_total_u": np.sum(np.abs(np.diff(u))),
        "u_min": np.min(u),
        "u_max": np.max(u),
    }


resultados = []
for i, idx_global in enumerate(INDICES):
    m = metricas_trayectoria(X[i], U[i])
    m["trayectoria"] = idx_global
    resultados.append(m)

df_metricas = pd.DataFrame(resultados).set_index("trayectoria")
display(df_metricas)

## 10. Tiempo de estabilización

Se deja un criterio configurable. Una trayectoria se considera estabilizada cuando, desde cierto instante \(k\), **todos** los estados permanecen dentro de sus umbrales.

Los valores siguientes son provisionales y deben ajustarse a las unidades y restricciones del sistema original.

In [ ]:
UMBRAL_ESTADO = np.array([
    0.02,  # |x|
    0.02,  # |x_dot|
    0.02,  # |theta|
    0.02,  # |theta_dot|
])


def tiempo_estabilizacion(x, thresholds=UMBRAL_ESTADO, dt=1.0):
    dentro = np.all(np.abs(x) <= thresholds, axis=1)

    for k in range(len(dentro)):
        if np.all(dentro[k:]):
            return k * dt

    return np.nan


filas = []
for i, idx_global in enumerate(INDICES):
    filas.append({
        "trayectoria": idx_global,
        "tiempo_estabilizacion": tiempo_estabilizacion(X[i])
    })

df_estabilizacion = pd.DataFrame(filas).set_index("trayectoria")
display(df_estabilizacion)

## 11. Inspección de posible saturación del actuador

In [ ]:
print("Mínimo u observado:", U.min())
print("Máximo u observado:", U.max())

plt.figure(figsize=(8, 4))
plt.hist(U.ravel(), bins=50)
plt.xlabel("u")
plt.ylabel("Frecuencia")
plt.title("Distribución de acciones de control en la muestra cargada")
plt.grid(True, alpha=0.3)
plt.show()

print(
    "\nNota: no se asumen todavía los límites exactos del actuador. "
    "Deben confirmarse desde el código/paper original."
)

## 12. Cargar un bloque mayor del dataset

Para análisis estadístico de cientos o miles de trayectorias se pueden cargar únicamente las primeras `N_FILAS`.

In [ ]:
def cargar_primeras_filas(path, nrows, dtype=np.float32):
    return pd.read_csv(
        path,
        header=None,
        nrows=nrows,
        dtype=dtype
    ).to_numpy()


# Ejemplo:
# N_FILAS = 1000
# X_block_flat = cargar_primeras_filas(X_PATH, N_FILAS)
# U_block = cargar_primeras_filas(U_PATH, N_FILAS)
# X0_block = cargar_primeras_filas(X0_PATH, N_FILAS)
# X_block = X_block_flat.reshape(N_FILAS, 26, 4)
# print(X_block.shape, U_block.shape, X0_block.shape)

## 13. Estadísticas globales por bloques

In [ ]:
def resumen_control_por_chunks(path, chunksize=5000, max_chunks=None):
    minimo = np.inf
    maximo = -np.inf
    suma = 0.0
    suma2 = 0.0
    n = 0

    for chunk_idx, chunk in enumerate(
        pd.read_csv(path, header=None, chunksize=chunksize, dtype=np.float32)
    ):
        a = chunk.to_numpy()
        minimo = min(minimo, float(a.min()))
        maximo = max(maximo, float(a.max()))
        suma += float(a.sum())
        suma2 += float((a.astype(np.float64)**2).sum())
        n += a.size

        if max_chunks is not None and (chunk_idx + 1) >= max_chunks:
            break

    media = suma / n
    var = suma2 / n - media**2
    std = np.sqrt(max(var, 0.0))

    return {
        "n_valores": n,
        "u_min": minimo,
        "u_max": maximo,
        "u_media": media,
        "u_std": std,
    }


# Ejemplo:
# resumen_u = resumen_control_por_chunks(U_PATH, chunksize=5000, max_chunks=10)
# resumen_u

## 14. Preparación para comparar MPC vs Red Neuronal

La siguiente etapa incorporará el modelo neuronal publicado.

El flujo será:

\[
x_k
\rightarrow
\mathrm{NN}
\rightarrow
u_k^{NN}
\rightarrow
\mathrm{dinámica\ Cart\mbox{-}Pole}
\rightarrow
x_{k+1}
\]

y se comparará contra \(u_k^{MPC}\).

In [ ]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred)**2))


def mae(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs(y_true - y_pred))


def max_abs_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.max(np.abs(y_true - y_pred))


def comparar_controladores(u_mpc, u_nn):
    return {
        "RMSE_u": rmse(u_mpc, u_nn),
        "MAE_u": mae(u_mpc, u_nn),
        "MaxAbsError_u": max_abs_error(u_mpc, u_nn),
        "VariacionTotal_MPC": np.sum(np.abs(np.diff(u_mpc))),
        "VariacionTotal_NN": np.sum(np.abs(np.diff(u_nn))),
    }


# Ejemplo futuro:
# metricas_nn = comparar_controladores(U[0], U_NN)
# metricas_nn

## 15. Tabla final prevista para la tesis

| Modelo | RMSE u vs MPC | Estable | Tiempo estabilización | Variación u | Flash | SRAM | Latencia |
|---|---:|---|---:|---:|---:|---:|---:|
| Float32 | ... | Sí/No | ... | ... | ... | ... | ... |
| PTQ INT8 | ... | Sí/No | ... | ... | ... | ... | ... |
| QAT INT8 | ... | Sí/No | ... | ... | ... | ... | ... |
| INT16x8 | ... | Sí/No | ... | ... | ... | ... | ... |
| Mixed | ... | Sí/No | ... | ... | ... | ... | ... |

Esta estructura separa:
- **fidelidad matemática** respecto al MPC;
- **comportamiento dinámico en lazo cerrado**;
- **costo de hardware** en STM32N6570-DK.